### test set eval - LR baseline & CNN

setup:

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import signal
from scipy.integrate import simpson

SEED = 1738
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA = Path('eeg_starter/data')
MODEL_FILE = Path('runs/k15_full_overlap.pt')
FS, C = 64, 20

def metrics(log_pred, age):
    pred = np.exp(log_pred)
    return {'RMSE': np.sqrt(np.mean((pred - age) ** 2)), 'MAE': np.mean(np.abs(pred - age))}

data:

In [2]:
WIN, STRIDE, N_SAMPLES = 8 * FS, 4 * FS, 896 * FS
CH_BASELINE = 14 # C3-P3
WIN_B, N_WIN_B, WELCH_S = 30 * FS, 10, 4
BANDS = [(0.5, 4), (4, 8), (8, 12), (12, 30)]

def baseline_features(eeg):
    w = eeg[CH_BASELINE, :N_WIN_B * WIN_B].double().numpy().reshape(N_WIN_B, WIN_B)
    var_x, var_dx = w.var(1), np.diff(w, axis=1).var(1)
    f, psd = signal.welch(w, FS, nperseg=WELCH_S * FS, axis=1)
    bp = [simpson(psd[:, (f >= lo) & (f <= hi)], dx=f[1] - f[0], axis=1) for lo, hi in BANDS]
    with np.errstate(all='ignore'):
        feats = np.stack([np.log(var_x), np.sqrt(var_dx / var_x), *np.log(bp)], axis=1)
    feats[var_x < 1e-6] = np.nan
    return np.nanmedian(feats, axis=0)

def load(name, cnn_windows):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    S = torch.empty(len(seg), C, N_SAMPLES, dtype=torch.float16) if cnn_windows else None
    feats = []
    for i, s in enumerate(seg):
        eeg = torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))
        feats.append(baseline_features(eeg))
        if cnn_windows:
            S[i] = eeg[:, :N_SAMPLES]
    return dict(age=np.array([s[3] for s in seg]), feats=np.stack(feats),
                X=S.unfold(2, WIN, STRIDE).transpose(1, 2) if cnn_windows else None)

data = {'train': load('train', cnn_windows=False), 'val': load('val', cnn_windows=True), 'test': load('test', cnn_windows=True)}
for name, d in data.items():
    print(name, len(d['age']), 'subjects', '' if d['X'] is None else tuple(d['X'].shape))

train 825 subjects 
val 103 subjects (103, 223, 20, 512)
test 104 subjects (104, 223, 20, 512)


CNN:

In [3]:
class CNN(nn.Module):
    def __init__(self, kernels, filters, pool):
        super().__init__()
        layers, f_in = [], 1
        for k, f in zip(kernels, filters):
            layers += [nn.Conv1d(C * f_in, C * f, k, groups=C), nn.ReLU()]
            f_in = f
        self.conv = nn.Sequential(*layers, nn.MaxPool1d(pool), nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.fc = nn.Linear(C * filters[-1], 1)

    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return np.array([model(x.float().to(DEVICE)).mean().item() for x in X])

saved = torch.load(MODEL_FILE, weights_only=False)
cfg = saved['config']
cnn = CNN(cfg['kernels'], cfg['filters'], cfg['pool']).to(DEVICE)
cnn.load_state_dict(saved['state_dict'])

pred_cnn = {split: predict(cnn, data[split]['X']) for split in ['val', 'test']}

baseline

In [4]:
mu, sd = data['train']['feats'].mean(0), data['train']['feats'].std(0)
D = lambda X: np.column_stack([np.ones(len(X)), (X - mu) / sd])
theta, *_ = np.linalg.lstsq(D(data['train']['feats']), np.log(data['train']['age']), rcond=None)
pred_lin = {split: D(data[split]['feats']) @ theta for split in ['val', 'test']}

results

In [5]:
mean_log_age = np.log(data['train']['age']).mean()
models = {'Mean of log(age)': {s: np.full(len(data[s]['age']), mean_log_age) for s in ['val', 'test']}, 'Linear baseline': pred_lin, 'CNN': pred_cnn}

results = pd.DataFrame({(name, split): metrics(pred[split], data[split]['age']) for name, pred in models.items() for split in ['val', 'test']}).T
results.index.names = ['model', 'split']
results.xs('test', level='split').loc[['Linear baseline', 'CNN']].rename_axis(None).round(3)

,RMSE,MAE
Linear baseline,2.422,1.483
CNN,0.991,0.612


blud

test set errors by age

In [6]:
age = data['test']['age']
groups = [(0, 1), (1, 3), (3, 8), (8, 17)]
rows = {}
for name in ['Linear baseline', 'CNN']:
    p = models[name]['test']
    for lo, hi in groups:
        m = (age >= lo) & (age < hi)
        rows[(name, f'{lo}-{hi} y (n={m.sum()})')] = {'MAE (y)': np.mean(np.abs(np.exp(p[m]) - age[m])), 'mean log error': np.mean(p[m] - np.log(age[m]))}
pd.DataFrame(rows).T.round(3)

MAE (y)  mean log error
Linear baseline 0-1 y (n=31)     0.467           0.385
                1-3 y (n=22)     1.128           0.271
                3-8 y (n=28)     1.446          -0.127
                8-17 y (n=23)    3.236          -0.183
CNN             0-1 y (n=31)     0.138           0.123
                1-3 y (n=22)     0.216           0.042
                3-8 y (n=28)     0.833          -0.028
                8-17 y (n=23)    1.360           0.066

age distribution for each set

In [7]:
sets = {s: data[s]['age'] for s in ['train', 'val', 'test']}
sets['all'] = np.concatenate(list(sets.values()))
for name, a in sets.items():
    print(f"{name:5s}  n {len(a):4d}  mean {a.mean():.2f}  SD {a.std(ddof=1):.2f}  median {np.median(a):.2f}")

train  n  825  mean 4.59  SD 4.30  median 2.98
val    n  103  mean 3.78  SD 3.99  median 2.07
test   n  104  mean 4.48  SD 4.36  median 2.63
all    n 1032  mean 4.50  SD 4.28  median 2.81


### Report table

Validation and test set RMSE in years.

In [8]:
results['RMSE'].unstack('split').reindex(list(models))[['val', 'test']].add_suffix(' RMSE (years)').rename_axis(index=None, columns=None).round(3)

,val RMSE (years),test RMSE (years)
Mean of log(age),4.139,4.719
Linear baseline,1.623,2.422
CNN,0.820,0.991
